# 01 · Pipeline runner

Runs the movie pipeline from a notebook and shows what happened. It calls the same code as
`python run_pipeline.py`, so a run started here is recorded in `pipeline_runs` exactly like a scheduled one.

**Stages:** `all` (extract → validate + transform → load → refresh model), or one of
`extract`, `transform`, `load`, `refresh`.

Change the parameters in the next cell, then *Run All*. With papermill the same cell is overridden from
the command line:

```
papermill notebooks/01_pipeline_runner.ipynb notebooks/reports/run.ipynb -p STAGE load
```

In [ ]:
# Parameters (papermill overrides these)
STAGE = "all"            # all | extract | transform | load | refresh
FULL_REFRESH = False     # True re-fetches every movie and forces a model rebuild
RUN_DATE = None          # YYYY-MM-DD for STAGE transform or load (None means latest)

In [ ]:
import sys
from pathlib import Path

# Find the project root (the folder that contains pipeline/config.py) so the notebook
# works from Jupyter, from papermill and from Task Scheduler alike.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pipeline" / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import sqlite3
from contextlib import closing

import pandas as pd

from pipeline import config, run_pipeline

print(f"Project root : {ROOT}")
print(f"Python       : {sys.executable}")
print(f"Database     : {config.DB_PATH}")

## Run the pipeline

Progress lines are also written to `logs/pipeline.log`. If a stage fails, the error is shown here and the run is recorded as `failed`.

In [ ]:
summary = run_pipeline.run(stage=STAGE, full_refresh=FULL_REFRESH, run_date=RUN_DATE)
pd.Series(summary, name="this run").to_frame()

## Latest runs

The newest row is the run above.

In [ ]:
with closing(sqlite3.connect(config.DB_PATH)) as conn:
    runs = pd.read_sql(
        "SELECT run_id, started_at, stage, full_refresh, status, duration_seconds, rows_extracted, "
        "rows_rejected, rows_loaded, rows_inserted, rows_updated, model_refreshed, error_message "
        "FROM pipeline_runs ORDER BY run_id DESC LIMIT 10", conn)
    totals = {t: conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
              for t in ("movies", "genres", "cast_crew", "trailers")}
runs

In [ ]:
pd.Series(totals, name="rows in the curated database").to_frame()